# Module 28 — Siamese BERT for Semantic Similarity

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Transformers, PyTorch, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 27 (Word2Vec), Module 4 (PyTorch Tensors)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Relevance Dataset](#3-synthetic-relevance-dataset)
4. [BERT Model Setup](#4-bert-model-setup)
5. [Siamese Architecture](#5-siamese-architecture)
6. [Training](#6-training)
7. [Evaluation](#7-evaluation)
8. [Visualization](#8-visualization)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why Siamese BERT for semantic similarity?

Siamese networks learn **deep semantic representations**:
- **Contextual embeddings**: BERT understands word context.
- **Transfer learning**: Pre-trained on massive text corpus.
- **Fine-tuning**: Adapt to domain-specific similarity tasks.

**Applications at Yandex**:
1. **Semantic search**: Match queries to documents by meaning.
2. **Duplicate detection**: Find similar questions/content.
3. **Recommendation**: Match users to relevant content.

**Key advantages over Word2Vec**:
- **Context-aware**: Same word has different embeddings in different contexts.
- **Pre-trained**: Leverages knowledge from massive corpora.
- **Fine-tunable**: Can adapt to specific similarity tasks.

In this notebook we will:
1. Generate synthetic query-document relevance pairs.
2. Build a Siamese BERT architecture.
3. Fine-tune with contrastive loss.
4. Evaluate on held-out relevance data.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Deep learning ────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports loaded")
print(f"  Device: {device}")

---
## §3 · Synthetic Relevance Dataset

In [ ]:
# Generate synthetic query-document relevance pairs
N_PAIRS = 5000

# Query-document templates
templates = [
    {'query': 'best laptop for programming', 'relevant': 'top laptops for developers 2024', 'irrelevant': 'best restaurants in moscow'},
    {'query': 'how to learn python', 'relevant': 'python programming tutorial for beginners', 'irrelevant': 'best hiking trails in california'},
    {'query': 'cheap flights to london', 'relevant': 'affordable airfare to london uk', 'irrelevant': 'recipe for chocolate cake'},
    {'query': 'iphone 15 review', 'relevant': 'apple iphone 15 detailed review', 'irrelevant': 'history of ancient rome'},
    {'query': 'yoga for beginners', 'relevant': 'beginner yoga poses and routines', 'irrelevant': 'stock market analysis today'},
]

# Generate pairs
pairs = []
for _ in range(N_PAIRS):
    template = np.random.choice(templates)
    pairs.append({
        'query': template['query'],
        'document': template['relevant'],
        'label': 1  # Relevant
    })
    pairs.append({
        'query': template['query'],
        'document': template['irrelevant'],
        'label': 0  # Not relevant
    })

df = pd.DataFrame(pairs)

print(f"✓ Generated {len(df)} relevance pairs")
print(f"  Relevant: {df['label'].sum()}")
print(f"  Not relevant: {(1 - df['label']).sum()}")
print(f"\nSample pairs:")
for i in range(3):
    print(f"  Query: {df['query'].iloc[i]}")
    print(f"  Document: {df['document'].iloc[i]}")
    print(f"  Label: {'Relevant' if df['label'].iloc[i] == 1 else 'Not relevant'}")
    print()

---
## §4 · BERT Model Setup

In [ ]:
# Load pre-trained BERT tokenizer and model
print("Loading BERT model...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Move to device
bert_model = bert_model.to(device)

print(f"✓ BERT model loaded")
print(f"  Model: bert-base-uncased")
print(f"  Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

# Test tokenization
test_text = "best laptop for programming"
tokens = tokenizer(test_text, return_tensors='pt', padding=True, truncation=True)
print(f"\nTokenization example:")
print(f"  Text: {test_text}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")

---
## §5 · Siamese Architecture

In [ ]:
# Define Siamese BERT model
class SiameseBERT(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model
        self.projection = nn.Linear(768, 128)
    
    def forward_one(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        projected = self.projection(cls_embedding)
        return projected
    
    def forward(self, input_ids_1, mask_1, input_ids_2, mask_2):
        embedding_1 = self.forward_one(input_ids_1, mask_1)
        embedding_2 = self.forward_one(input_ids_2, mask_2)
        return embedding_1, embedding_2

# Create model
model = SiameseBERT(bert_model).to(device)

print(f"✓ Siamese BERT model created")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## §6 · Training

In [ ]:
# Define contrastive loss
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, embedding_1, embedding_2, label):
        distance = torch.pairwise_distance(embedding_1, embedding_2)
        loss = label * distance.pow(2) + \
               (1 - label) * torch.clamp(self.margin - distance, min=0).pow(2)
        return loss.mean()

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = ContrastiveLoss(margin=1.0)

# Prepare data (simplified for demo)
N_EPOCHS = 3
BATCH_SIZE = 16

print(f"Training setup:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: 1e-5")
print(f"  Loss: Contrastive Loss")
print(f"\nNote: Full training would take ~30 minutes on GPU")
print(f"This is a simplified demo showing the architecture.")

---
## §7 · Evaluation

In [ ]:
# Evaluate model (simplified)
def compute_similarity(query, document, model, tokenizer, device):
    """Compute similarity between query and document."""
    # Tokenize
    query_tokens = tokenizer(query, return_tensors='pt', padding=True, truncation=True).to(device)
    doc_tokens = tokenizer(document, return_tensors='pt', padding=True, truncation=True).to(device)
    
    # Get embeddings
    with torch.no_grad():
        query_emb = model.forward_one(query_tokens['input_ids'], query_tokens['attention_mask'])
        doc_emb = model.forward_one(doc_tokens['input_ids'], doc_tokens['attention_mask'])
    
    # Compute cosine similarity
    similarity = torch.cosine_similarity(query_emb, doc_emb)
    return similarity.item()

# Test similarity (using pre-trained BERT, not fine-tuned)
test_pairs = [
    ('best laptop for programming', 'top laptops for developers'),
    ('best laptop for programming', 'best restaurants in moscow'),
    ('how to learn python', 'python programming tutorial'),
    ('how to learn python', 'history of ancient rome'),
]

print("=" * 70)
print("SEMANTIC SIMILARITY EXAMPLES")
print("=" * 70)

for query, doc in test_pairs:
    sim = compute_similarity(query, doc, model, tokenizer, device)
    print(f"\nQuery: {query}")
    print(f"Document: {doc}")
    print(f"Similarity: {sim:.4f}")

---
## §8 · Visualization

In [ ]:
# Visualize similarity scores
similarities = []
labels = []

for query, doc in test_pairs:
    sim = compute_similarity(query, doc, model, tokenizer, device)
    similarities.append(sim)
    labels.append(f"{query[:20]}... ↔ {doc[:20]}...")

fig = go.Figure(data=[
    go.Bar(
        x=labels,
        y=similarities,
        marker_color=['#00CC96' if '↔' in l and l.split('↔')[0].strip()[:10] == l.split('↔')[1].strip()[:10] else '#EF553B' for l in labels]
    )
])

fig.update_layout(
    title='Semantic Similarity Scores',
    xaxis_title='Query-Document Pairs',
    yaxis_title='Cosine Similarity',
    height=400
)
fig.show()

print("💡 Key insights:")
print("   - BERT captures semantic meaning beyond keywords")
print("   - Similar queries have higher similarity scores")
print("   - Fine-tuning would improve domain-specific performance")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Search & NLP Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Semantic search, duplicate detection, relevance ranking |
> | **Model** | BERT-base for most tasks, larger models for complex queries |
> | **Training** | Fine-tune on domain-specific relevance data |
> | **Evaluation** | MRR, NDCG, semantic similarity metrics |
> | **Deployment** | Pre-compute embeddings, use FAISS for fast retrieval |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Hybrid search architecture**:
>    ```
>    Query → BM25 retrieval (fast) → BERT re-ranking (accurate)
>    ```
>
> 2. **Use cases**:
>    | Use Case | Model Size | Latency |
>    |----------|------------|---------|
>    | Real-time search | DistilBERT | < 50ms |
>    | Batch re-ranking | BERT-large | < 500ms |
>    | Offline indexing | BERT-large | N/A |
>
> 3. **Production tips**:
>    - Use model distillation for faster inference
>    - Cache embeddings for frequent queries
>    - A/B test against BM25 baseline
>
> ### 🔑 Key Takeaways
>
> 1. **BERT captures deep semantic meaning** beyond keyword matching.
> 2. **Siamese architecture** enables efficient similarity computation.
> 3. **Fine-tuning on domain data** significantly improves performance.
> 4. **Hybrid search** (BM25 + BERT) provides best results.
> 5. **Model distillation** is essential for production deployment.